# Lab 6.1 – Umsatz prognostizieren

**Starter.** Kopie in `project/playground/` bearbeiten. Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

| Prognose | Formel | Status |
| --- | --- | --- |
| naiv | Umsatz im selben Monat des Vorjahres | fertig |
| einfach | Vorjahresmonat × Trendfaktor; Trendfaktor = Umsatz der letzten 12 Monate / Umsatz der 12 Monate davor | Aufgabe 1 |

Ablauf: Monatsumsatz → beide Prognosen → Backtest April–September 2026 mit MAPE → Prognose Oktober–Dezember 2026. Ergebnis: `prognose_2026.csv` und `prognose_2026.html` in `ORDNER`.

In [ ]:
import pandas as pd
import plotly.express as px

EINGABE = "project/examples/m02-data-preparation/l03-features/umsatz_features.csv"
ORDNER = "project/playground/"     # hier landen prognose_2026.csv und prognose_2026.html

df = pd.read_csv(EINGABE)
umsatz = df.groupby("monat")["umsatz"].sum().round(2)    # Monatsumsatz: "2024-01" -> Betrag
print(len(umsatz), "Monate von", umsatz.index[0], "bis", umsatz.index[-1])
px.line(x=umsatz.index, y=umsatz.values, markers=True,
        title="Monatsumsatz: jedes Jahr dasselbe Saisonmuster, das Niveau steigt",
        labels={"x": "Monat", "y": "Umsatz in €"})

## Schritt 1 (fertig): naive Prognose

`stand` ist der letzte Monat, dessen Umsatz bekannt ist. Die naive Prognose braucht ihn nicht, die einfache Prognose schon (Trendfaktor bis einschließlich `stand`).

In [ ]:
def vorjahresmonat(monat):
    """'2026-04' -> '2025-04'"""
    return str(int(monat[:4]) - 1) + monat[4:]


def prognose_naiv(umsatz, monat, stand):
    """Naive Prognose: Umsatz im selben Monat des Vorjahres (stand wird hier nicht gebraucht)."""
    return umsatz[vorjahresmonat(monat)]


print("Naive Prognose für 2026-10:", round(prognose_naiv(umsatz, "2026-10", "2026-09")))

## Schritt 2: einfache Prognose mit Trendfaktor (Aufgabe 1)

Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 3) füllen, dann die Prüfzelle ausführen.

In [ ]:
# >>> AUFGABE 1: Funktion prognose_einfach(umsatz, monat, stand) schreiben – Prompt siehe Übung, Aufgabe 3

# <<< ENDE AUFGABE 1

In [ ]:
# Prüfzelle – nicht ändern
def pruefe_prognose():
    if "prognose_einfach" not in globals():
        return "nicht OK – Funktion prognose_einfach fehlt noch (Aufgabe 1)."
    april = prognose_einfach(umsatz, "2026-04", "2026-03")
    if abs(april - 288113.12) > 1:
        return (f"nicht OK – Prognose 2026-04 mit Stand 2026-03 ist {april:,.0f} statt 288,113. "
                "Trendfaktor = Summe der 12 Monate bis einschließlich stand / Summe der 12 Monate davor.")
    oktober = prognose_einfach(umsatz, "2026-10", "2026-09")
    if abs(oktober - 354163.88) > 1:
        return f"nicht OK – Prognose 2026-10 mit Stand 2026-09 ist {oktober:,.0f} statt 354,164: stand wird nicht verwendet."
    return "OK – 2026-04: 288.113 € (Vorjahr 272.033 € × Trendfaktor 1,059); 2026-10: 354.164 €"


print(pruefe_prognose())

## Schritt 3 (fertig): Backtest April–September 2026

So tun, als wäre heute Ende März 2026 (`STAND`): beide Prognosen für sechs Monate rechnen, deren Ergebnis inzwischen bekannt ist. Fehler in % je Monat; MAPE = Mittelwert der Fehlerbeträge („im Schnitt x % daneben“).

In [ ]:
STAND = "2026-03"
TESTMONATE = ["2026-04", "2026-05", "2026-06", "2026-07", "2026-08", "2026-09"]

backtest = pd.DataFrame({"monat": TESTMONATE})
backtest["ist"] = [umsatz[m] for m in TESTMONATE]
backtest["naiv"] = [prognose_naiv(umsatz, m, STAND) for m in TESTMONATE]
backtest["einfach"] = [prognose_einfach(umsatz, m, STAND) for m in TESTMONATE]
backtest["fehler_naiv_prozent"] = (backtest["naiv"] / backtest["ist"] - 1) * 100
backtest["fehler_einfach_prozent"] = (backtest["einfach"] / backtest["ist"] - 1) * 100

mape_naiv = backtest["fehler_naiv_prozent"].abs().mean()
mape_einfach = backtest["fehler_einfach_prozent"].abs().mean()
print(f"MAPE naiv:    {mape_naiv:.1f} %")
print(f"MAPE einfach: {mape_einfach:.1f} %")
backtest.round(1)

In [ ]:
px.line(backtest, x="monat", y=["ist", "naiv", "einfach"], markers=True,
        title=f"Backtest: einfache Prognose im Schnitt {mape_einfach:.1f} % daneben, naive {mape_naiv:.1f} %",
        labels={"monat": "Monat", "value": "Umsatz in €", "variable": ""})

## Schritt 4 (fertig): Prognose Oktober–Dezember 2026

Jetzt mit Stand September 2026, also mit allen bekannten Monaten.

In [ ]:
STAND = "2026-09"
ZUKUNFT = ["2026-10", "2026-11", "2026-12"]

prognose = pd.DataFrame({"monat": ZUKUNFT})
prognose["naiv"] = [prognose_naiv(umsatz, m, STAND) for m in ZUKUNFT]
prognose["einfach"] = [prognose_einfach(umsatz, m, STAND) for m in ZUKUNFT]
prognose = prognose.round(0)
prognose.to_csv(ORDNER + "prognose_2026.csv", index=False)
print("Gespeichert:", ORDNER + "prognose_2026.csv")
prognose

In [ ]:
verlauf = pd.DataFrame({"monat": umsatz.index, "umsatz": umsatz.values, "art": "Ist"})
vorschau = pd.DataFrame({"monat": ["2026-09"] + ZUKUNFT,                 # 2026-09 verbindet beide Linien
                         "umsatz": [umsatz["2026-09"]] + list(prognose["einfach"]), "art": "Prognose (einfach)"})
fig = px.line(pd.concat([verlauf, vorschau]), x="monat", y="umsatz", color="art", markers=True,
              title=f"Prognose Q4 2026: {prognose['einfach'].sum() / 1e6:.2f} Mio. € (Backtest: im Schnitt {mape_einfach:.1f} % daneben)",
              labels={"monat": "Monat", "umsatz": "Umsatz in €", "art": ""})
fig.write_html(ORDNER + "prognose_2026.html", include_plotlyjs="cdn")
fig

## Erweiterung: Spanne oder Backtest je Region

Prompt und Lösung: Übung, Erweiterung.

In [ ]:
# Erweiterung: Spanne ± MAPE oder Backtest je Region (siehe Übung)
